# Customer Churn & Revenue Impact Analysis

This notebook is my analysis on a telecom churn dataset. The goal is to understand why customers churn and how much revenue loss it causes.

I'll go through:
- Loading and cleaning the data
- Some basic EDA
- Visualizations
- Revenue impact calculation

In [ ]:
# importing the libraries I need
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# just setting a style so plots look decent
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (8, 5)

## Step 1: Load the Data

In [ ]:
df = pd.read_csv('../data/telecom_churn.csv')

print('Shape of dataset:', df.shape)
df.head()

In [ ]:
# checking what columns we have and their types
df.info()

In [ ]:
# how many nulls in each column
df.isnull().sum()

## Step 2: Data Cleaning

TotalCharges has some blank spaces which cause issues, need to fix that

In [ ]:
# TotalCharges is sometimes stored as string with spaces
# converting it to numeric and replacing errors with NaN
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# checking how many NaN we got
print('Nulls in TotalCharges:', df['TotalCharges'].isnull().sum())

In [ ]:
# dropping rows where TotalCharges is null
# these are usually new customers with 0 tenure so not much loss
df = df.dropna(subset=['TotalCharges'])

print('Dataset shape after dropping nulls:', df.shape)

In [ ]:
# converting Churn column to 0 and 1 for easier analysis
df['Churn_Binary'] = df['Churn'].map({'Yes': 1, 'No': 0})

# quick check
df[['Churn', 'Churn_Binary']].value_counts()

## Step 3: Basic EDA

Let's see the overall churn rate and some patterns

In [ ]:
# overall churn rate
churn_rate = df['Churn_Binary'].mean() * 100
print(f'Overall Churn Rate: {churn_rate:.2f}%')

# counts
print('\nChurn counts:')
print(df['Churn'].value_counts())

In [ ]:
# churn by contract type - I was curious if this matters
contract_churn = df.groupby('Contract')['Churn_Binary'].mean() * 100
print('Churn rate by Contract type (%):')
print(contract_churn.sort_values(ascending=False))

In [ ]:
# some basic stats for churned vs non-churned customers
df.groupby('Churn')[['tenure', 'MonthlyCharges', 'TotalCharges']].mean().round(2)

## Step 4: Visualizations

In [ ]:
# Churn distribution - simple bar chart
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# plot 1: churn count
churn_counts = df['Churn'].value_counts()
axes[0].bar(churn_counts.index, churn_counts.values, color=['steelblue', 'tomato'])
axes[0].set_title('Churn Count')
axes[0].set_xlabel('Churn')
axes[0].set_ylabel('Number of Customers')
for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v + 50, str(v), ha='center', fontweight='bold')

# plot 2: churn by contract type
contract_churn_count = df.groupby(['Contract', 'Churn']).size().unstack()
contract_churn_count.plot(kind='bar', ax=axes[1], color=['steelblue', 'tomato'], edgecolor='white')
axes[1].set_title('Churn by Contract Type')
axes[1].set_xlabel('Contract')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=15)
axes[1].legend(['No Churn', 'Churned'])

plt.tight_layout()
plt.savefig('../images/churn_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved!')

In [ ]:
# boxplot for monthly charges - churned vs not churned
# wanted to see if price affects churn

plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x='Churn', y='MonthlyCharges', palette={'No': 'steelblue', 'Yes': 'tomato'})
plt.title('Monthly Charges: Churned vs Not Churned')
plt.xlabel('Churned?')
plt.ylabel('Monthly Charges ($)')
plt.savefig('../images/monthly_charges_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved!')

In [ ]:
# tenure distribution for churned customers
# hypothesis: newer customers churn more

plt.figure(figsize=(9, 5))
df[df['Churn'] == 'Yes']['tenure'].hist(bins=20, color='tomato', alpha=0.7, label='Churned')
df[df['Churn'] == 'No']['tenure'].hist(bins=20, color='steelblue', alpha=0.6, label='Not Churned')
plt.title('Tenure Distribution by Churn Status')
plt.xlabel('Tenure (months)')
plt.ylabel('Count')
plt.legend()
plt.tight_layout()
plt.show()

## Step 5: Revenue Impact

Now let me estimate how much revenue is being lost due to churn

In [ ]:
# total customers who churned
churned_customers = df[df['Churn'] == 'Yes']

print(f'Total churned customers: {len(churned_customers)}')
print(f'Avg monthly charges of churned customers: ${churned_customers["MonthlyCharges"].mean():.2f}')

# estimated monthly revenue loss
monthly_revenue_loss = churned_customers['MonthlyCharges'].sum()
print(f'\nEstimated monthly revenue loss: ${monthly_revenue_loss:,.2f}')

# annual revenue loss estimate
annual_revenue_loss = monthly_revenue_loss * 12
print(f'Estimated annual revenue loss: ${annual_revenue_loss:,.2f}')

In [ ]:
# Revenue loss by contract type
revenue_by_contract = churned_customers.groupby('Contract')['MonthlyCharges'].sum()
print('Monthly revenue loss by contract type:')
print(revenue_by_contract)

In [ ]:
# customer segments based on tenure
# I'm splitting into 3 simple groups

def tenure_segment(t):
    if t <= 12:
        return 'New (0-12 months)'
    elif t <= 36:
        return 'Mid (13-36 months)'
    else:
        return 'Loyal (36+ months)'

df['TenureSegment'] = df['tenure'].apply(tenure_segment)

segment_churn = df.groupby('TenureSegment')['Churn_Binary'].mean() * 100
print('Churn rate by tenure segment (%):')
print(segment_churn.sort_values(ascending=False))

## Summary of Findings

Based on the analysis:

1. **~26% overall churn rate** - roughly 1 in 4 customers leaves
2. **Month-to-month contract customers churn the most** - probably because there's no commitment
3. **Churned customers pay higher monthly charges on average** - this hurts more since high-paying customers are leaving
4. **New customers (0-12 months) are at highest risk** - first year is critical
5. **Estimated annual revenue loss is significant** - could be millions depending on company size

These insights can help the business target retention efforts at the right customers.